In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

valid_df = pd.read_csv("../data/valid_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)

/root/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/miniconda3/compiler_compat/ld: warning: librt.so.1, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/root/miniconda3/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
valid_df = pd.read_csv("../data/valid_rewrite_001.csv")
valid_id_2_gold_citations_d = {}
valid_id_2_query_d = {}
for query_id, gold_citations, query in zip(valid_df['query_id'], valid_df['gold_citations'], valid_df['query2']):
    valid_id_2_gold_citations_d[query_id] = gold_citations.split(';')
    valid_id_2_query_d[query_id] = query

In [5]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

valid_multi_question_df = pd.read_csv("../data/valid_rewrite_split_question_001.csv")

recall_citations_l = []
gold_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(valid_multi_question_df['query_id'], valid_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in valid_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

recall_hits_l = []
for query_id, query_list in query_id_2_query_list.items():
    all_hits = []
    for query in query_list:
        hits1 = dense_index.search_with_score(query, top_k=500)
        hits2 = sparse_index.search_with_score(query, top_k=500)
        hits_merge = hits_utils.merge_hits_with_score_l_by_max(hits1, hits2)
        all_hits = hits_utils.merge_hits_with_score_l_by_max(all_hits, hits_merge)
        
    print(f"[{query_id}] hits_merge.len:", len(all_hits))
    gold_citations_l.append(valid_id_2_gold_citations_d[query_id])

    # citations = [hit['citation'] for hit, score in all_hits]

    recall_hits_l.append([hit for hit, score in all_hits])
    
    second_layer = citation_utils.compute_citation_score_with_sentence_pos(all_hits, decay="log")

    citations = []
    for citation, score in second_layer:
        if citation in court_consideration_d:
            citations.append(citation)
        if citation in law_d:
            citations.append(citation)

    recall_citations_l.append(list(set(citations)))
    
r = metric_utils.cal_recall(recall_citations_l, gold_citations_l)
print(r)

# 准备好document
# recall_hits_l = []
# for recall_citations in recall_citations_l:
#     recall_hits = []
#     for citation in set(recall_citations):
#         if citation in court_consideration_d:
#             recall_hits.append({'citation':citation, 'text':court_consideration_d[citation]})
#         elif citation in law_d:
#             recall_hits.append({'citation':citation, 'text':law_d[citation]})
#     recall_hits_l.append(recall_hits)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[val_001] hits_merge.len: 1260
[val_002] hits_merge.len: 2300
[val_003] hits_merge.len: 2073
[val_004] hits_merge.len: 1627
[val_005] hits_merge.len: 2129
[val_006] hits_merge.len: 1648
[val_007] hits_merge.len: 2548
[val_008] hits_merge.len: 3432
[val_009] hits_merge.len: 1961
[val_010] hits_merge.len: 2147
0.5025175903029323


In [6]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):
    print("===>",query_id)
    ranked_citation_l_l = []

    raw_query = query_list[-1]
    normalized_query_list = query_list[:-1] # 最后一个包含多个问题
    question_l = []
    for q in normalized_query_list:
        _, question = q.rsplit('\n\n', 1)
        question_l.append(question)

    # 先算context和recall_hits的分数
    # context, _ = query_list[0].rsplit('\n\n', 1)
    # hit_with_score_l_context = reranker_utils.rerank_by_batch_chunked2(reranker, context, recall_hits_l[idx])
    # assert(len(hit_with_score_l_context) == len(recall_hits_l[idx]))

    hit_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, recall_hits_l[idx])
    hit_with_score_l_question = reranker_utils.rerank_by_batch_chunked2(reranker, '\n\n'.join(question_l), recall_hits_l[idx])
    
    
    hit_with_score_l = [(raw_score[0], raw_score[1]+0.5*question_score[1]) for raw_score, question_score in zip(hit_with_score_l_raw, hit_with_score_l_question)]

    # citation_score_l = citation_utils.compute_citation_score_with_court_consideration_sector_pos(hit_with_score_l)
    citation_score_l = citation_utils.compute_citation_score_with_sentence_pos(hit_with_score_l, 'log')

    # 对综合分数排序
    sorted_citation_score_l = sorted([(c,score) for c,score in citation_score_l], key=lambda x: x[1], reverse=True)
    
    # 将citation合并
    query_result = [citation for citation, _ in sorted_citation_score_l]
    
    query_result2 = [citation for citation in query_result if citation in court_consideration_d or citation in law_d]
    
    result_citation_l.append(query_result2)

    print("result_citation_l[-1].len:", len(result_citation_l[-1]))
    
max_limit = None
max_r = None
max_p = None
max_f1 = 0
for limit in [5,10,15,20,25,30,35,40,45]:
    result_citation_l2 = [r[:limit] for r in result_citation_l]
    result = metric_utils.macro_f1(result_citation_l2, gold_citations_l[:len(result_citation_l2)])
    if max_f1 < result['macro_f1']:
        max_r = result['macro_recall']
        max_p = result['macro_precision']
        max_limit = limit
        max_f1 = result['macro_f1']
print(f"[{max_limit}] r:", max_r, ", p:", max_p, "f1:",max_f1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


  0%|          | 0/10 [00:00<?, ?it/s]

===> val_001


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


result_citation_l[-1].len: 259
===> val_002
result_citation_l[-1].len: 306
===> val_003
result_citation_l[-1].len: 589
===> val_004
result_citation_l[-1].len: 610
===> val_005
result_citation_l[-1].len: 809
===> val_006
result_citation_l[-1].len: 776
===> val_007
result_citation_l[-1].len: 1409
===> val_008
result_citation_l[-1].len: 1519
===> val_009
result_citation_l[-1].len: 615
===> val_010
result_citation_l[-1].len: 850
[15] r: 0.1814214004965057 , p: 0.24 f1: 0.1950635185592224
